# **Pull GitHub Repository**

In [1]:
!pip install -q torchmetrics timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 927.3/927.3 kB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 77.2 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

!rm -rf /content/ECM3401_Individual_Project

token = userdata.get('GitHub')
!git clone -b causal https://{token}@github.com/sccthomas/ECM3401_Individual_Project.git

Cloning into 'ECM3401_Individual_Project'...
remote: Enumerating objects: 2091, done.
remote: Counting objects: 100% (252/252), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 2091 (delta 162), reused 146 (delta 75), pack-reused 1839 (from 2)
Receiving objects: 100% (2091/2091), 22.59 MiB | 14.41 MiB/s, done.
Resolving deltas: 100% (1425/1425), done.


In [3]:
import sys

sys.path.append('/content/ECM3401_Individual_Project/')
!ls /content/ECM3401_Individual_Project/src/

dataset  __init__.py  self_supervised_learning	training  vision_transformer


In [4]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


# **Define the Model**

In [6]:
import torch
from src.vision_transformer.model import SemanticSegmentationVisionTransformer
from src.self_supervised_learning.contrastive_loss import ContrastivePreTraining
from src.self_supervised_learning.masked_region_loss import MaskedRegionLoss

# --------------------------------------------
# Parameters
# --------------------------------------------
device = torch.device("cuda")
metric_device = torch.device("cpu")

image_dims = (3, 256, 256)  # Input image dimensions
patch_embedding_scale_1 = (16, 768)  # Patch size and embedding dimension for scale 1
patch_embedding_scale_2 = (8, 512)  # Patch size and embedding dimension for scale 2
patch_embedding_scale_3 = (4, 256)  # Patch size and embedding dimension for scale 3

# --------------------------------------------
# Model Initialization
# --------------------------------------------
model = SemanticSegmentationVisionTransformer(
    # - Image dimensions
    image_dims=image_dims,
    # - Hyper Parameters
    num_encoder_layers=6,
    use_heavyweight_decoder=False,
    use_swin_transformer=False,
    use_skip_layer_gated_attention=False,
    use_learnable_skip_layers=True,
    skip_layer_ratio=2,
    encoder_dropout_rate=0.2,
    patch_fusion_dropout_rate=0.2,
    decoder_dropout_rate=0.2,
    num_encoder_heads=8,
    num_classes=1,
    # - Scales
    patch_embedding_scale_1=patch_embedding_scale_1,
    patch_embedding_scale_2=patch_embedding_scale_2,
    patch_embedding_scale_3=patch_embedding_scale_3,
).to(device)

ssl_model = MaskedRegionLoss(
    model=model,
    max_patch_size=16,
).to(device)

# ssl_model = ContrastivePreTraining(
#     model=model,
#     encoder_dims=[768, 512, 256],
#     projection_dim=128,
# ).to(device)

In [7]:
ssl_model

MaskedRegionLoss(
  (_SelfSupervisedLoss__model): SemanticSegmentationVisionTransformer(
    (_SemanticSegmentationVisionTransformer__patch_embedding_modules): ModuleDict(
      (x1): PatchEmbedding(
        (_PatchEmbedding__projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (x2): PatchEmbedding(
        (_PatchEmbedding__projection): Conv2d(3, 512, kernel_size=(8, 8), stride=(8, 8))
      )
      (x3): PatchEmbedding(
        (_PatchEmbedding__projection): Conv2d(3, 256, kernel_size=(4, 4), stride=(4, 4))
      )
    )
    (_SemanticSegmentationVisionTransformer__encoders): ModuleDict(
      (x1): ModuleList(
        (0-5): 6 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (linear1): Linear(in_features=768, out_features=1536, bias=True)
          (dropout): Dropout(p=0.2, inplace=False)
          (linear2): Lin

In [8]:
model

SemanticSegmentationVisionTransformer(
  (_SemanticSegmentationVisionTransformer__patch_embedding_modules): ModuleDict(
    (x1): PatchEmbedding(
      (_PatchEmbedding__projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (x2): PatchEmbedding(
      (_PatchEmbedding__projection): Conv2d(3, 512, kernel_size=(8, 8), stride=(8, 8))
    )
    (x3): PatchEmbedding(
      (_PatchEmbedding__projection): Conv2d(3, 256, kernel_size=(4, 4), stride=(4, 4))
    )
  )
  (_SemanticSegmentationVisionTransformer__encoders): ModuleDict(
    (x1): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (linear1): Linear(in_features=768, out_features=1536, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=1536, out_features=768, bias=True)
        (norm1): LayerNorm((768,), eps=

# **Train the Model With SSL**

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from src.dataset.snow import SnowDataset
from src.training.self_supervised_learning import train_model

# --------------------------------------------
# Parameters
# --------------------------------------------
dataset_dir = "/content/drive/MyDrive/snow_dataset"  # Replace with your dataset path
batch_size = 32
num_epochs = 50
learning_rate = 1e-4
patience = 5  # Early stopping patience

# --------------------------------------------
# Dataset and DataLoader
# --------------------------------------------

# Load the dataset and split it into train and validation sets
train_dataset = SnowDataset(
    dataset_dir_path=dataset_dir,
    normalize=True,
    resize=True,
    rotate=True,
)
print("Length of dataset:", len(train_dataset))
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=12,
    persistent_workers=True,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=12,
    persistent_workers=True,
    pin_memory=True,
)
# --------------------------------------------
# Loss, Optimizer, and Scheduler
# --------------------------------------------
optimizer = optim.AdamW(ssl_model.parameters(), lr=learning_rate, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=num_epochs // 2, T_mult=1)

# --------------------------------------------
# Mixed Precision Setup
# --------------------------------------------
scaler = torch.amp.GradScaler(device.type)

# --------------------------------------------
# Train Model
# --------------------------------------------
train_model(
    ssl_model=ssl_model,
    num_epochs=num_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    train_loader=train_loader,
    val_loader=val_loader,
    patience=patience,
    device=device,
)

Length of dataset: 18209

Epoch 1/50


Training: 100%|██████████| 456/456 [31:38<00:00,  4.16s/it]


Training Loss: 0.8337


Validation: 100%|██████████| 114/114 [04:35<00:00,  2.42s/it]


Validation Loss: 0.2404

Epoch 2/50


...

Training: 100%|██████████| 456/456 [10:37<00:00,  1.40s/it]


Training Loss: 0.0812


Validation: 100%|██████████| 114/114 [00:47<00:00,  2.38it/s]


Validation Loss: 0.0765

Epoch 14/50


Training:  30%|███       | 138/456 [03:15<07:23,  1.39s/it]

In [ ]:
!cp /content/best_model_ssl.pth /content/drive/MyDrive/best_ssl_mea_model.pth

In [ ]:
!cp /content/best_model.pth /content/drive/MyDrive/best_model_ssl_mea.pth

# **Train the Model Fine Tuning**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from src.dataset.snow import SnowDataset
from src.training.train import train_model
import os as _os

# --------------------------------------------
# Environment
# --------------------------------------------
_os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --------------------------------------------
# Parameters
# --------------------------------------------
dataset_dir = "/content/drive/MyDrive/snow_dataset"  # Replace with your dataset path
batch_size = 32
num_epochs = 20
learning_rate = 1e-4
patience = 5  # Early stopping patience

# --------------------------------------------
# Dataset and DataLoader
# --------------------------------------------

# Load the dataset and split it into train and validation sets
train_dataset = SnowDataset(
    dataset_dir_path=dataset_dir,
    resize=True,
    normalize=True,
    rotate=True,
)
print("Length of dataset:", len(train_dataset))
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=12,
    persistent_workers=True,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=12,
    persistent_workers=True,
    pin_memory=True,
)

# --------------------------------------------
# Loss, Optimizer, and Scheduler
# --------------------------------------------
criterion = nn.BCEWithLogitsLoss()  # Binary cross-entropy loss with logits
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)  # Use CosineAnnealingLR scheduler

# --------------------------------------------
# Mixed Precision Setup
# --------------------------------------------
scaler = torch.amp.GradScaler(device.type)

# --------------------------------------------
# Train Model
# --------------------------------------------
train_model(
    model=model,
    num_epochs=num_epochs,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    train_loader=train_loader,
    val_loader=val_loader,
    patience=patience,
    device=device,
    metric_device=metric_device,
)

In [ ]:
!cp /content/best_model.pth /content/drive/MyDrive/best_model.pth

# **Evaluate the Model**

In [ ]:
import torch

# Load the model's state_dict
path = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/trained_models/contrastive_ssl/model.pth"
# path = "/content/drive/MyDrive/best_model_ssl.pth"
checkpoint = torch.load(path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint)
model = model.eval()

# Load the SSL model's state_dict
path = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/trained_models/contrastive_ssl/ssl_model.pth"
# path = "/content/drive/MyDrive/best_ssl_contrastive_model.pth"
checkpoint = torch.load(path, map_location=device, weights_only=True)
ssl_model.load_state_dict(checkpoint)
ssl_model = ssl_model.eval()

In [ ]:
from src.dataset.snow import SnowDataset
from torch.utils.data import DataLoader

dataset_dir = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/snow_dataset"
# dataset_dir = "/content/drive/MyDrive/snow_dataset"

# Dataset and DataLoader
batch_size = 5

train_dataset = SnowDataset(
    dataset_dir_path=dataset_dir,
    len_override=30,
    resize=True,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=1,
)
images, masks = next(iter(train_loader))
images, masks = images.to(device), masks.to(device)

In [ ]:
from src.training.evaluation import evaluate_with_no_modifications, evaluate_with_illumination_modifications, \
    evaluate_with_background_modifications, evaluate_with_texture_modifications

idx = 2

In [ ]:
evaluate_with_no_modifications(model, images[idx], masks[idx], device)

In [ ]:
evaluate_with_illumination_modifications(model, images[idx], masks[idx], device)

In [ ]:
evaluate_with_background_modifications(model, images[idx], masks[idx], device, mtype="Sharpness")

In [ ]:
evaluate_with_texture_modifications(model, images[idx], masks[idx], device, texture_type="Cellular Variability")

## **Contrastive SSL**

In [ ]:
from src.self_supervised_learning.contrastive_loss import visualize_tsne

visualize_tsne(ssl_model, images)